# IndicTrans2 Bidirectional Fine-tuning: English ↔ Odia

This notebook fine-tunes **both translation directions** using LoRA:
- **Direction A:** English → Odia (using `indictrans2-en-indic-1B`)
- **Direction B:** Odia → English (using `indictrans2-indic-en-1B`)

**Key points:**
- IndicTrans2 uses separate model checkpoints per direction — we load and fine-tune BOTH
- Each direction gets its own LoRA adapter
- `forced_bos_token_id` ensures correct output script (Odia / English)
- Script verification after every translation batch

> ⚠️ **Run Step 0 first, then use Runtime → Restart and run all (or restart kernel before Step 1)**

## Step 0: Install Dependencies
**After this cell finishes → RESTART THE RUNTIME, then continue from Step 1**

In [ ]:
# 1. Update core libraries (Wait for this to finish)
!pip install -q -U transformers==4.45.2 tokenizers==0.20.1
!pip install -q -U numpy>=2.1 torch>=2.5
!pip install -q sacrebleu sentencepiece peft accelerate

# Install IndicTransToolkit (provides IndicProcessor for preprocessing)
!pip install -q indictranstoolkit

print('\n✅ Done. NOW RESTART THE RUNTIME before continuing!')
print('   Runtime → Restart session  (Colab)  or  Kernel → Restart (Jupyter)')

In [ ]:
!pip install "numpy<2"
!pip install "transformers==4.38.2"
!pip install "peft==0.10.0"
!pip install "accelerate==0.27.2"

---
## Step 1: Load Dataset

In [2]:
import pandas as pd
#!pip install google-colab

# ── Mount Google Drive if using Colab ──
#from google.colab import drive
#drive.mount('/content/drive')

DATA_PATH = '/kaggle/input/datasets/peeyoushh/train-final/train.final'  # <-- change to your actual path

df = pd.read_csv(DATA_PATH, sep='\t', header=None, names=['source', 'english', 'odia'])
df = df.dropna(subset=['english', 'odia']).reset_index(drop=True)
print(f'Total rows: {len(df)}')
print(df.head(2))

Total rows: 23789
    source                                            english  \
0  genesis  In the beginning God created the heaven and th...   
1  genesis  And the earth was without form, and void; and ...   

                                                odia  
0       ଆରମ୍ଭରେ ପରମେଶ୍ବର ଆକାଶ ଓ ପୃଥିବୀକୁ ସୃଷ୍ଟି କଲେ।  
1  ପୃଥିବୀ ସେତବେେଳେ ସଂପୂରନ୍ଭାବେ ଶୂନ୍ଯ ଓ କିଛି ନଥିଲା...  


In [3]:
# Verify Odia script (Odia = U+0B00–U+0B7F, Devanagari = U+0900–U+097F)
def script_of(text):
    for ch in text:
        if '\u0B00' <= ch <= '\u0B7F': return 'Odia'
        if '\u0900' <= ch <= '\u097F': return 'Devanagari'
    return 'Other'

scripts = df['odia'].apply(script_of)
print('Script distribution in your dataset:')
print(scripts.value_counts())
print('\nExpected: mostly Odia. If Devanagari — fix your dataset first!')

Script distribution in your dataset:
odia
Odia          23786
Other             2
Devanagari        1
Name: count, dtype: int64

Expected: mostly Odia. If Devanagari — fix your dataset first!


In [4]:
# Split: 10000 train, 300 eval
TRAIN_SIZE = 10000
EVAL_SIZE  = 300

df = df.sample(frac=1, random_state=42).reset_index(drop=True)
train_df = df.iloc[:TRAIN_SIZE].reset_index(drop=True)
eval_df  = df.iloc[TRAIN_SIZE:TRAIN_SIZE + EVAL_SIZE].reset_index(drop=True)

print(f'Train: {len(train_df)}, Eval: {len(eval_df)}')
print(f'\nBidirectional training will use the SAME parallel pairs in both directions:')
print(f'  Direction A (EN→OR): english column → odia column')
print(f'  Direction B (OR→EN): odia column → english column')

Train: 10000, Eval: 300

Bidirectional training will use the SAME parallel pairs in both directions:
  Direction A (EN→OR): english column → odia column
  Direction B (OR→EN): odia column → english column


---
## Step 2: Load Both Models + IndicProcessor

IndicTrans2 uses **separate checkpoints** for each direction:
- `ai4bharat/indictrans2-en-indic-1B` → English to Indic
- `ai4bharat/indictrans2-indic-en-1B` → Indic to English

We load both for bidirectional training.

In [ ]:
# Fix numpy version if needed
!pip install "numpy<2"

In [ ]:
!pip install -q indictranstoolkit

In [ ]:
!pip install "transformers==4.38.2"

In [ ]:
# Upgrade transformers to ensure the correct internal paths exist
!pip install --upgrade transformers

In [ ]:
!pip uninstall -y transformers IndicTransToolkit
!pip install transformers IndicTransToolkit

In [ ]:
try:
    from transformers import PreTrainedTokenizerBase
    print("Success! Transformers is working correctly.")
except ImportError as e:
    print(f"Still failing: {e}")

In [5]:
# ── IndicProcessor import with fallback ──
try:
    from IndicTransToolkit import IndicProcessor
    print('Imported IndicProcessor from IndicTransToolkit')
except ImportError:
    from IndicTransToolkit.IndicTransToolkit import IndicProcessor
    print('Imported IndicProcessor via fallback path')

import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

2026-03-03 15:17:47.794155: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772551067.822568     512 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772551067.831774     512 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772551067.855792     512 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772551067.855838     512 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772551067.855842     512 computation_placer.cc:177] computation placer alr

Imported IndicProcessor from IndicTransToolkit
Using device: cuda


### Step 2a: Load English → Odia Model

In [ ]:
from kaggle_secrets import UserSecretsClient
secret_label = "HF_TOKEN"
secret_value = UserSecretsClient().get_secret(secret_label)

In [6]:
from huggingface_hub import login

# When you run this, a text box will appear. Paste your token there.
login()

In [39]:
# ───── DIRECTION A: English → Odia ─────
EN2OR_MODEL_NAME = 'ai4bharat/indictrans2-en-indic-1B'
EN2OR_SRC_LANG   = 'eng_Latn'
EN2OR_TGT_LANG   = 'ory_Orya'   # Odia script

print('Loading EN→OR tokenizer...')
en2or_tokenizer = AutoTokenizer.from_pretrained(EN2OR_MODEL_NAME, trust_remote_code=True)

# Verify ory_Orya token exists
ory_orya_id = en2or_tokenizer.convert_tokens_to_ids(EN2OR_TGT_LANG)
print(f'Token "{EN2OR_TGT_LANG}" → ID: {ory_orya_id}')
assert ory_orya_id != en2or_tokenizer.unk_token_id, \
    f'ERROR: ory_Orya mapped to UNK! Check tokenizer.'
print('✅ ory_Orya token confirmed.')

print('\nLoading EN→OR base model...')
en2or_base_model = AutoModelForSeq2SeqLM.from_pretrained(
    EN2OR_MODEL_NAME,
    trust_remote_code=True,
    torch_dtype=torch.float16 if device == 'cuda' else torch.float32,
).to(device)
en2or_base_model.eval()

en2or_ip = IndicProcessor(inference=True)
print('✅ EN→OR model loaded.')

Loading EN→OR tokenizer...
Token "ory_Orya" → ID: 69
✅ ory_Orya token confirmed.

Loading EN→OR base model...
✅ EN→OR model loaded.


### Step 2b: Load Odia → English Model

In [18]:
# ───── DIRECTION B: Odia → English ─────
OR2EN_MODEL_NAME = 'ai4bharat/indictrans2-indic-en-1B'
OR2EN_SRC_LANG   = 'ory_Orya'
OR2EN_TGT_LANG   = 'eng_Latn'

print('Loading OR→EN tokenizer...')
or2en_tokenizer = AutoTokenizer.from_pretrained(OR2EN_MODEL_NAME, trust_remote_code=True)

# Verify eng_Latn token exists
eng_latn_id = or2en_tokenizer.convert_tokens_to_ids(OR2EN_TGT_LANG)
print(f'Token "{OR2EN_TGT_LANG}" → ID: {eng_latn_id}')
assert eng_latn_id != or2en_tokenizer.unk_token_id, \
    f'ERROR: eng_Latn mapped to UNK! Check tokenizer.'
print('✅ eng_Latn token confirmed.')

print('\nLoading OR→EN base model...')
or2en_base_model = AutoModelForSeq2SeqLM.from_pretrained(
    OR2EN_MODEL_NAME,
    trust_remote_code=True,
    torch_dtype=torch.float16 if device == 'cuda' else torch.float32,
).to(device)
or2en_base_model.eval()

or2en_ip = IndicProcessor(inference=True)
print('✅ OR→EN model loaded.')

Loading OR→EN tokenizer...
Token "eng_Latn" → ID: 4
✅ eng_Latn token confirmed.

Loading OR→EN base model...
✅ OR→EN model loaded.


### Quick Translation Tests (Both Directions)

In [ ]:
# ── Test EN → Odia ──
test_en = [
    "God created the heaven and the earth.",
    "The light was good.",
    "Write your own sentence here."
]

preprocessed = en2or_ip.preprocess_batch(test_en, src_lang=EN2OR_SRC_LANG, tgt_lang=EN2OR_TGT_LANG)
inputs = en2or_tokenizer(preprocessed, truncation=True, padding='longest',
                         max_length=256, return_tensors='pt').to(device)

with torch.no_grad():
    outputs = en2or_base_model.generate(
        **inputs, num_beams=4, max_length=256,
        forced_bos_token_id=ory_orya_id,
    )

translated = en2or_tokenizer.batch_decode(outputs, skip_special_tokens=True,
                                          clean_up_tokenization_spaces=True)
translated = en2or_ip.postprocess_batch(translated, lang=EN2OR_TGT_LANG)

print('=== EN → Odia Test ===\n')
for eng, odi in zip(test_en, translated):
    print(f'EN : {eng}')
    print(f'OR : {odi}  [{script_of(odi)}]\n')

In [ ]:
# ── Test Odia → EN ──
test_or = [
    "ଆରମ୍ଭରେ ପରମେଶ୍ବର ଆକାଶ ଓ ପୃଥିବୀକୁ ସୃଷ୍ଟି କଲେ।",
    "ଆଲୋକ ଭଲ ଥିଲା।",
    "ଏଠାରେ ନିଜର ବାକ୍ୟ ଲେଖନ୍ତୁ।"
]

preprocessed = or2en_ip.preprocess_batch(test_or, src_lang=OR2EN_SRC_LANG, tgt_lang=OR2EN_TGT_LANG)
inputs = or2en_tokenizer(preprocessed, truncation=True, padding='longest',
                         max_length=256, return_tensors='pt').to(device)

with torch.no_grad():
    outputs = or2en_base_model.generate(
        **inputs, num_beams=4, max_length=256,
        forced_bos_token_id=eng_latn_id,
    )

translated = or2en_tokenizer.batch_decode(outputs, skip_special_tokens=True,
                                          clean_up_tokenization_spaces=True)
translated = or2en_ip.postprocess_batch(translated, lang=OR2EN_TGT_LANG)

print('=== Odia → EN Test ===\n')
for odi, eng in zip(test_or, translated):
    print(f'OR : {odi}')
    print(f'EN : {eng}\n')

---
## Step 3: Baseline BLEU (Both Directions, Before Fine-tuning)

In [40]:
from sacrebleu.metrics import BLEU, CHRF

def translate_batch(sentences, model, tokenizer, ip_proc, src_lang, tgt_lang,
                    forced_bos_id, batch_size=8):
    """Generalized translate function for either direction."""
    model.eval()
    all_out = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i : i + batch_size]

        preprocessed = ip_proc.preprocess_batch(batch, src_lang=src_lang, tgt_lang=tgt_lang)

        inputs = tokenizer(
            preprocessed, truncation=True, padding='longest',
            max_length=256, return_tensors='pt'
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                num_beams=4,
                max_length=256,
                forced_bos_token_id=forced_bos_id,
            )

        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True,
                                         clean_up_tokenization_spaces=True)
        decoded = ip_proc.postprocess_batch(decoded, lang=tgt_lang)
        all_out.extend(decoded)

    return all_out


def score(hyps, refs):
    bleu = BLEU(tokenize='char').corpus_score(hyps, [refs])
    chrf = CHRF(word_order=2).corpus_score(hyps, [refs])   # chrF++
    return bleu, chrf


# Use 100 examples for speed
eval_en  = eval_df['english'].tolist()[:100]
eval_or  = eval_df['odia'].tolist()[:100]

In [41]:
# ── Baseline: EN → Odia ──
print('Evaluating baseline EN→OR...')
baseline_en2or_hyp = translate_batch(
    eval_en, en2or_base_model, en2or_tokenizer, en2or_ip,
    EN2OR_SRC_LANG, EN2OR_TGT_LANG, ory_orya_id
)
baseline_en2or_bleu, baseline_en2or_chrf = score(baseline_en2or_hyp, eval_or)
print(f'\n📊 BASELINE EN→OR  →  BLEU: {baseline_en2or_bleu.score:.2f}   chrF++: {baseline_en2or_chrf.score:.2f}')

print('\nSample (script check):')
for eng, ref, hyp in zip(eval_en[:3], eval_or[:3], baseline_en2or_hyp[:3]):
    print(f'  EN : {eng[:70]}')
    print(f'  REF: {ref[:60]}  [{script_of(ref)}]')
    print(f'  OUT: {hyp[:60]}  [{script_of(hyp)}]')
    print()

Evaluating baseline EN→OR...

📊 BASELINE EN→OR  →  BLEU: 30.12   chrF++: 23.56

Sample (script check):
  EN : And he said unto them, I must preach the kingdom of God to other citie
  REF: କିନ୍ତୁ ଯୀଶୁ ସମାନଙ୍କେୁ କହିଲେ, "ପରମେଶ୍ବରଙ୍କ ରାଜ୍ଯର ସୁସମାଚାର ମା  [Odia]
  OUT: ମରିଯୁ ସେ ସେମାନଙ୍କୁ କହିଲେ, ମୋତେ ଅନ୍ଯ଼ ନଗରମାନଙ୍କରେ ମଧ୍ଯ଼ ପରମେଶ  [Odia]

  EN : And Mizraim begat Ludim, and Anamim, and Lehabim, and Naphtuhim
  REF: ମିଶ୍ରଯୀମ (ମିଶର), ଲୂଦୀଯ, ଅନାମୀଯ, ଲହାବୀଯ ଓ ନପ୍ତୂହୀଯମାନଙ୍କର ପିତ  [Odia]
  OUT: ମରିଯୁ ମିଜ୍ରାଇମ୍ ଲୁଦିମ୍, ଅନାମିମ୍, ଲହାବିମ୍, ଏବଂ ନପ୍ତୁହିମ୍ଙ୍କୁ   [Odia]

  EN : They reap every one his corn in the field: and they gather the vintage
  REF: ଗରିବ ଲୋକମାନେ ବହୁତ ବିଳମ୍ବିତ ରାତ୍ରିୟାଏ ଫସଲ ଅମଳ କରନ୍ତି। ସମେନେ   [Odia]
  OUT: ମରିଯୁ ସେମାନେ ସମସ୍ତେ କ୍ଷେତରେ ନିଜ ନିଜ ଶସ୍ଯ଼ ଅମଳ କରନ୍ତି। ଏବଂ ସେ  [Odia]



In [42]:
# ── Baseline: Odia → EN ──
print('Evaluating baseline OR→EN...')
baseline_or2en_hyp = translate_batch(
    eval_or, or2en_base_model, or2en_tokenizer, or2en_ip,
    OR2EN_SRC_LANG, OR2EN_TGT_LANG, eng_latn_id
)
baseline_or2en_bleu, baseline_or2en_chrf = score(baseline_or2en_hyp, eval_en)
print(f'\n📊 BASELINE OR→EN  →  BLEU: {baseline_or2en_bleu.score:.2f}   chrF++: {baseline_or2en_chrf.score:.2f}')

print('\nSample:')
for odi, ref, hyp in zip(eval_or[:3], eval_en[:3], baseline_or2en_hyp[:3]):
    print(f'  OR : {odi[:60]}')
    print(f'  REF: {ref[:70]}')
    print(f'  OUT: {hyp[:70]}')
    print()

Evaluating baseline OR→EN...

📊 BASELINE OR→EN  →  BLEU: 44.71   chrF++: 38.43

Sample:
  OR : କିନ୍ତୁ ଯୀଶୁ ସମାନଙ୍କେୁ କହିଲେ, "ପରମେଶ୍ବରଙ୍କ ରାଜ୍ଯର ସୁସମାଚାର ମା
  REF: And he said unto them, I must preach the kingdom of God to other citie
  OUT: .But Jesus said to them, "I must preach the good news of the kingdom o

  OR : ମିଶ୍ରଯୀମ (ମିଶର), ଲୂଦୀଯ, ଅନାମୀଯ, ଲହାବୀଯ ଓ ନପ୍ତୂହୀଯମାନଙ୍କର ପିତ
  REF: And Mizraim begat Ludim, and Anamim, and Lehabim, and Naphtuhim
  OUT: .Mishrayim (Egypt), was the father of the Lydians, the Anamites, the L

  OR : ଗରିବ ଲୋକମାନେ ବହୁତ ବିଳମ୍ବିତ ରାତ୍ରିୟାଏ ଫସଲ ଅମଳ କରନ୍ତି। ସମେନେ 
  REF: They reap every one his corn in the field: and they gather the vintage
  OUT: . The poor harvest very late at night. Surely the rich gather grapes f



---
## Step 4: Fine-tune Both Directions with LoRA

We train **two separate LoRA adapters** — one for each direction.

In [ ]:
!pip install "peft==0.10.0"
!pip install "accelerate==0.27.2"

In [10]:
from torch.utils.data import Dataset
from peft import get_peft_model, LoraConfig, TaskType

class BiDirectionalDataset(Dataset):
    """Generalized dataset for either translation direction."""
    def __init__(self, src_list, tgt_list, tokenizer, src_lang, tgt_lang, max_len=256):
        self.src = src_list
        self.tgt = tgt_list
        self.tokenizer = tokenizer
        self.src_lang = src_lang
        self.tgt_lang = tgt_lang
        self.ip_train = IndicProcessor(inference=False)
        self.max_len = max_len

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        src_text = self.ip_train.preprocess_batch(
            [self.src[idx]], src_lang=self.src_lang, tgt_lang=self.tgt_lang)[0]
        tgt_text = self.ip_train.preprocess_batch(
            [self.tgt[idx]], src_lang=self.tgt_lang, tgt_lang=self.src_lang)[0]

        model_in = self.tokenizer(
            src_text, truncation=True, max_length=self.max_len,
            padding='max_length', return_tensors='pt'
        )
        with self.tokenizer.as_target_tokenizer():
            label_enc = self.tokenizer(
                tgt_text, truncation=True, max_length=self.max_len,
                padding='max_length', return_tensors='pt'
            )

        labels = label_enc['input_ids'].squeeze().clone()
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            'input_ids':      model_in['input_ids'].squeeze(),
            'attention_mask': model_in['attention_mask'].squeeze(),
            'labels':         labels,
        }


# LoRA config (shared hyperparams for both directions)
# lora_cfg = LoraConfig(
#     task_type=TaskType.SEQ_2_SEQ_LM,
#     r=8,
#     lora_alpha=16,
#     target_modules=['q_proj', 'v_proj', 'k_proj', 'out_proj', 'fc1', 'fc2'],
#     lora_dropout=0.1,
#     bias='none',
# )

print('✅ Dataset class and LoRA config ready.')

✅ Dataset class and LoRA config ready.


### Step 4a: Fine-tune English → Odia

In [ ]:
# ── EN→OR: Create dataset ──
en2or_train_dataset = BiDirectionalDataset(
    train_df['english'].tolist(),
    train_df['odia'].tolist(),
    en2or_tokenizer,
    EN2OR_SRC_LANG,
    EN2OR_TGT_LANG
)
print(f'EN→OR train dataset: {len(en2or_train_dataset)} examples')

In [ ]:
# ── EN→OR: Load fresh model + apply LoRA ──
en2or_ft_model = AutoModelForSeq2SeqLM.from_pretrained(
    EN2OR_MODEL_NAME, trust_remote_code=True, torch_dtype=torch.float32
).to(device)

en2or_ft_model.enable_input_require_grads()
en2or_ft_model = get_peft_model(en2or_ft_model, lora_cfg)
print('EN→OR LoRA:')
en2or_ft_model.print_trainable_parameters()

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

en2or_training_args = Seq2SeqTrainingArguments(
    output_dir='./it2_en2or_lora',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,    # effective batch = 16
    learning_rate=5e-5,
    warmup_steps=100,
    logging_steps=50,
    save_steps=500,
    save_total_limit=2,
    predict_with_generate=False,
    fp16=(device == 'cuda'),
    gradient_checkpointing=False,
    report_to='none',
)

en2or_trainer = Seq2SeqTrainer(
    model=en2or_ft_model,
    args=en2or_training_args,
    train_dataset=en2or_train_dataset,
    tokenizer=en2or_tokenizer,
    data_collator=DataCollatorForSeq2Seq(en2or_tokenizer, model=en2or_ft_model, pad_to_multiple_of=8),
)

print('\n🚀 Starting EN→OR fine-tuning... (~1-3 hrs on T4)')
en2or_trainer.train()
print('✅ EN→OR fine-tuning done!')

In [ ]:
# Save EN→OR LoRA adapter
en2or_ft_model.save_pretrained('./it2_en2or_lora_adapter')
en2or_tokenizer.save_pretrained('./it2_en2or_lora_adapter')
print('✅ EN→OR adapter saved to ./it2_en2or_lora_adapter')

### Step 4b: Fine-tune Odia → English

In [19]:
import torch, gc
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

# 8-bit setup
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_has_fp16_weight=False,
)

In [24]:
OR2EN_MODEL_NAME = 'ai4bharat/indictrans2-indic-en-1B'

print('Loading OR→EN tokenizer...')
or2en_tokenizer = AutoTokenizer.from_pretrained(OR2EN_MODEL_NAME, trust_remote_code=True)

print('\nLoading OR→EN base model with 8-bit Quantization on GPU 0...')
or2en_ft_model = AutoModelForSeq2SeqLM.from_pretrained(
    OR2EN_MODEL_NAME,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map={"": 0}  # 👈 THE FIX: Forces all layers onto cuda:0
)

Loading OR→EN tokenizer...

Loading OR→EN base model with 8-bit Quantization on GPU 0...


In [25]:
# ── OR→EN: Create dataset ──
or2en_train_dataset = BiDirectionalDataset(
    train_df['odia'].tolist(),     # source = Odia
    train_df['english'].tolist(),  # target = English
    or2en_tokenizer,
    OR2EN_SRC_LANG,
    OR2EN_TGT_LANG
)
print(f'OR→EN train dataset: {len(or2en_train_dataset)} examples')

OR→EN train dataset: 10000 examples


In [26]:
# Enable gradient checkpointing to save VRAM during training
or2en_ft_model.gradient_checkpointing_enable()

# Prepare for Int8 training
or2en_ft_model = prepare_model_for_kbit_training(or2en_ft_model)

# Define LoRA adapters
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "out_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_2_SEQ_LM"
)

# Apply adapters
or2en_ft_model = get_peft_model(or2en_ft_model, peft_config)
or2en_ft_model.config.use_cache = False  # Mandatory for training

print('✅ Model ready with trainable parameters!')
or2en_ft_model.print_trainable_parameters()

You are using an old version of the checkpointing format that is deprecated (We will also silently ignore `gradient_checkpointing_kwargs` in case you passed it).Please update to the new format on your modeling file. To use the new format, you need to completely remove the definition of the method `_set_gradient_checkpointing` in your model.
You are using an old version of the checkpointing format that is deprecated (We will also silently ignore `gradient_checkpointing_kwargs` in case you passed it).Please update to the new format on your modeling file. To use the new format, you need to completely remove the definition of the method `_set_gradient_checkpointing` in your model.


✅ Model ready with trainable parameters!
trainable params: 7,077,888 || all params: 1,030,084,608 || trainable%: 0.6871


In [ ]:
import gc
torch.cuda.empty_cache()
gc.collect()

In [ ]:
!pip install -U bitsandbytes

In [29]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

# Crucial for Seq2Seq QLoRA
or2en_ft_model.config.use_cache = False 

training_args = Seq2SeqTrainingArguments(
    output_dir="./indictrans2-qlora-out",
    per_device_train_batch_size=8,      # Stay at 1 for T4
    gradient_accumulation_steps=8,     # Total batch size 16
    learning_rate=2e-4,                 # QLoRA usually handles slightly higher LR
    num_train_epochs=3,
    logging_steps=10,
    fp16=True,
    optim="paged_adamw_8bit",          # Mandatory for stability
    save_total_limit=1,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=or2en_ft_model,
    args=training_args,
    train_dataset=or2en_train_dataset, # Ensure this is pre-processed with IndicProcessor
    tokenizer=or2en_tokenizer,
    data_collator=DataCollatorForSeq2Seq(or2en_tokenizer, model=or2en_ft_model, pad_to_multiple_of=8),
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4109: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: i

Step,Training Loss
10,2.427000
20,1.943200
30,1.600500
40,1.419700
50,1.343200
60,1.308500
70,1.321700
80,1.315600
90,1.244500
100,1.302500


TrainOutput(global_step=234, training_loss=1.375653552193927, metrics={'train_runtime': 7802.7718, 'train_samples_per_second': 3.845, 'train_steps_per_second': 0.03, 'total_flos': 4.008814294597632e+16, 'train_loss': 1.375653552193927, 'epoch': 2.9952})

In [37]:
import shutil

shutil.make_archive('it2_or2en_lora_adapter', 'zip', '/kaggle/working/it2_or2en_lora_adapter')

'/kaggle/working/it2_or2en_lora_adapter.zip'

In [36]:
# Save OR→EN LoRA adapter
or2en_ft_model.save_pretrained('./it2_or2en_lora_adapter')
or2en_tokenizer.save_pretrained('./it2_or2en_lora_adapter')
!cp -r ./it2_or2en_lora_adapter /content/drive/MyDrive/it2_or2en_lora_adapter
print('✅ OR→EN adapter saved to ./it2_or2en_lora_adapter')

cp: cannot create directory '/content/drive/MyDrive/it2_or2en_lora_adapter': No such file or directory
✅ OR→EN adapter saved to ./it2_or2en_lora_adapter


---
## Step 5: Post Fine-tuning Evaluation (Both Directions)

In [ ]:
# ── Evaluate fine-tuned EN→OR ──
en2or_ft_model.eval()
print('Evaluating fine-tuned EN→OR...')

ft_en2or_hyp = translate_batch(
    eval_en, en2or_ft_model, en2or_tokenizer, en2or_ip,
    EN2OR_SRC_LANG, EN2OR_TGT_LANG, ory_orya_id
)
ft_en2or_bleu, ft_en2or_chrf = score(ft_en2or_hyp, eval_or)
print(f'✅ Fine-tuned EN→OR  →  BLEU: {ft_en2or_bleu.score:.2f}   chrF++: {ft_en2or_chrf.score:.2f}')

In [38]:
# ── Evaluate fine-tuned OR→EN ──
or2en_ft_model.eval()
print('Evaluating fine-tuned OR→EN...')

ft_or2en_hyp = translate_batch(
    eval_or, or2en_ft_model, or2en_tokenizer, or2en_ip,
    OR2EN_SRC_LANG, OR2EN_TGT_LANG, eng_latn_id
)
ft_or2en_bleu, ft_or2en_chrf = score(ft_or2en_hyp, eval_en)
print(f'✅ Fine-tuned OR→EN  →  BLEU: {ft_or2en_bleu.score:.2f}   chrF++: {ft_or2en_chrf.score:.2f}')

Evaluating fine-tuned OR→EN...


/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


✅ Fine-tuned OR→EN  →  BLEU: 47.32   chrF++: 41.85


---
## Step 6: Combined Results + Sample Comparisons

In [ ]:
# ── Combined Results Table ──
print('=' * 70)
print('       BIDIRECTIONAL FINE-TUNING RESULTS')
print('=' * 70)
print(f'{"Direction":<12} | {"Metric":<8} | {"Baseline":>10} | {"Fine-tuned":>11} | {"Delta":>7}')
print('-' * 70)
print(f'{"EN→OR":<12} | {"BLEU":<8} | {baseline_en2or_bleu.score:10.2f} | {ft_en2or_bleu.score:11.2f} | {ft_en2or_bleu.score - baseline_en2or_bleu.score:+7.2f}')
print(f'{"EN→OR":<12} | {"chrF++":<8} | {baseline_en2or_chrf.score:10.2f} | {ft_en2or_chrf.score:11.2f} | {ft_en2or_chrf.score - baseline_en2or_chrf.score:+7.2f}')
print('-' * 70)
print(f'{"OR→EN":<12} | {"BLEU":<8} | {baseline_or2en_bleu.score:10.2f} | {ft_or2en_bleu.score:11.2f} | {ft_or2en_bleu.score - baseline_or2en_bleu.score:+7.2f}')
print(f'{"OR→EN":<12} | {"chrF++":<8} | {baseline_or2en_chrf.score:10.2f} | {ft_or2en_chrf.score:11.2f} | {ft_or2en_chrf.score - baseline_or2en_chrf.score:+7.2f}')
print('=' * 70)

In [ ]:
# ── Sample translations: EN → Odia ──
print('\n📝 EN → Odia Sample Comparisons:\n')
for i in range(5):
    print(f'[{i+1}] EN       : {eval_en[i][:75]}')
    print(f'     REF      : {eval_or[i][:65]}  [{script_of(eval_or[i])}]')
    print(f'     BASELINE : {baseline_en2or_hyp[i][:65]}  [{script_of(baseline_en2or_hyp[i])}]')
    print(f'     FINETUNED: {ft_en2or_hyp[i][:65]}  [{script_of(ft_en2or_hyp[i])}]')
    print()

In [ ]:
# ── Sample translations: Odia → EN ──
print('\n📝 Odia → EN Sample Comparisons:\n')
for i in range(5):
    print(f'[{i+1}] OR       : {eval_or[i][:65]}')
    print(f'     REF      : {eval_en[i][:75]}')
    print(f'     BASELINE : {baseline_or2en_hyp[i][:75]}')
    print(f'     FINETUNED: {ft_or2en_hyp[i][:75]}')
    print()

---
## Step 7: Save to Google Drive + Reload Instructions

### How to Reload the Fine-tuned Models Later

**English → Odia:**
```python
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from peft import PeftModel

base = AutoModelForSeq2SeqLM.from_pretrained('ai4bharat/indictrans2-en-indic-1B', trust_remote_code=True)
model = PeftModel.from_pretrained(base, './it2_en2or_lora_adapter')
model = model.merge_and_unload()  # merges LoRA into weights for faster inference
tokenizer = AutoTokenizer.from_pretrained('./it2_en2or_lora_adapter', trust_remote_code=True)
```

**Odia → English:**
```python
base = AutoModelForSeq2SeqLM.from_pretrained('ai4bharat/indictrans2-indic-en-1B', trust_remote_code=True)
model = PeftModel.from_pretrained(base, './it2_or2en_lora_adapter')
model = model.merge_and_unload()
tokenizer = AutoTokenizer.from_pretrained('./it2_or2en_lora_adapter', trust_remote_code=True)
```